# AWG engine


## 0. Driver check


In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")
    print(
        f"Max sample rate:  {spcm.Clock(card).sample_rate(max=True, return_unit=units.MHz)}"
    )

print("Driver check OK -- card opened, queried, and closed without errors.")

## 1. Setup


In [ ]:
import time

from awg_controller import (
    AODSettings,
    AWGBatch,
    AWGEngine,
    AWGEngineConfig,
    CardConfig,
    RFRamp,
)

CARD_PATH = "/dev/spcm0"
MAX_AMP_V = 1.0  # into 50 ohm
engine = None


def open_engine(cfg, f_lo, f_hi, grid):
    global engine
    if engine is not None:
        engine.close()
        engine = None
    aod = AODSettings(
        f_min_v=f_lo,
        f_max_v=f_hi,
        f_min_h=f_lo,
        f_max_h=f_hi,
        grid_rows=grid,
        grid_cols=grid,
    )
    card = CardConfig(card_path=CARD_PATH, max_amplitude_v=MAX_AMP_V, aod_settings=aod)
    assert card.max_amplitude_v <= 2.0, "exceeds hard safety ceiling"
    engine = AWGEngine(card, cfg)
    fs = engine.open()
    print(
        f"mode={cfg.mode}  fs={fs / 1e6:.1f} MHz  tones={grid}+{grid}  "
        f"Nyquist={fs / 2e6:.0f} MHz  tones {f_lo / 1e6:.1f}..{f_hi / 1e6:.1f} MHz"
    )
    print(f"max round = {engine.max_round_duration_s * 1e3:.1f} ms")
    return fs

## 2. Single ramp, STREAM mode


In [ ]:
OBSERVABLE_RAMP_S = 3.0
F_MIN, F_MAX = 90e6, 110e6

fs = open_engine(AWGEngineConfig(mode="stream"), F_MIN, F_MAX, grid=1)
print(f"ring = {engine.look_ahead_s * 1e3:.1f} ms of look-ahead")


def both_channels(f_start, f_end, duration_s, amp_pct=40.0):
    """One batch, one tone per channel, same ramp on each."""
    return AWGBatch(
        ramps=[
            RFRamp(
                channel=ch,
                f_start=f_start,
                f_end=f_end,
                amplitude_pct=amp_pct,
                tone_index=0,
            )
            for ch in (0, 1)
        ],
        travel_duration_s=duration_s,
    )


batches = [
    both_channels(F_MIN, F_MIN, 0.01),  # settle
    both_channels(F_MIN, F_MAX, OBSERVABLE_RAMP_S),  # the sweep
    both_channels(F_MAX, F_MAX, 0.01),  # define the resting frequency
]

engine.load_round(batches)
print(f"loaded {engine.total_travel_duration_s:.3f} s of waveform")

print("Playing -- watch the analyser now.")
engine.play()
assert engine.last_error is None, engine.last_error

time.sleep(OBSERVABLE_RAMP_S + 1.0)
assert engine.last_error is None, engine.last_error
print(f"Parked at {F_MAX / 1e6:.0f} MHz until close().")

## 3. One tweezer rotating around a 2x2 grid


In [ ]:
F_CENTER = 80e6
SPACING_HZ = 8e6

SITE_HZ = (F_CENTER - SPACING_HZ / 2, F_CENTER + SPACING_HZ / 2)

F_LO, F_HI = SITE_HZ

CYCLE = [(0, 0), (0, 1), (1, 1), (1, 0)]


def rotation_round(step_s, revolutions=4, amp_pct=20.0):
    def batch(row_from, row_to, col_from, col_to, duration_s):
        return AWGBatch(
            ramps=[
                RFRamp(
                    channel=0,
                    f_start=row_from,
                    f_end=row_to,
                    amplitude_pct=amp_pct,
                    tone_index=0,
                ),
                RFRamp(
                    channel=1,
                    f_start=col_from,
                    f_end=col_to,
                    amplitude_pct=amp_pct,
                    tone_index=0,
                ),
            ],
            travel_duration_s=duration_s,
        )

    row, col = (SITE_HZ[i] for i in CYCLE[0])

    batches = [batch(row, row, col, col, step_s)]  # settle at the first site
    for _ in range(revolutions):
        for k in range(1, len(CYCLE) + 1):
            row_to, col_to = (SITE_HZ[i] for i in CYCLE[k % len(CYCLE)])
            batches.append(batch(row, row_to, col, col_to, step_s))
            row, col = row_to, col_to

    batches.append(batch(row, row, col, col, step_s))  # rest where it started
    return batches

### 3a. Observable timescale -- STREAM mode


In [ ]:
fs = open_engine(AWGEngineConfig(mode="stream"), F_LO, F_HI, grid=1)

batches = rotation_round(step_s=0.5)
n_moving = sum(1 for b in batches if any(r.f_start != r.f_end for r in b.ramps))
print(f"{len(batches)} batches ({n_moving} moving), {len(batches[0].ramps)} ramps each")

engine.load_round(batches)
print(f"loaded {engine.total_travel_duration_s:.2f} s of waveform")

print("Playing -- one tone per channel, stepping a quarter cycle apart.")
engine.play()
assert engine.last_error is None, engine.last_error

time.sleep(engine.total_travel_duration_s + 1.0)
assert engine.last_error is None, engine.last_error
print("Rotation complete; tweezer parked at its start site until close().")

### 3b. Experiment timescale -- MEMORY mode


In [ ]:
cfg = AWGEngineConfig(
    mode="memory",
    sample_rate_hz=1.25e9,
    dma_buffer_samples=32 * 1024 * 1024,
    hold_tail_samples=1 << 20,
)
fs = open_engine(cfg, F_LO, F_HI, grid=1)

batches = rotation_round(step_s=5e-6)
round_s = sum(b.travel_duration_s for b in batches)
print(
    f"{len(batches)} batches, {round_s * 1e6:.0f} us round "
    f"({round_s / engine.max_round_duration_s * 100:.1f}% of the limit)"
)

t0 = time.perf_counter()
engine.load_round(batches)  # renders AND uploads, so this is the slow call
print(f"rendered + uploaded in {(time.perf_counter() - t0) * 1e3:.1f} ms")

engine.play()
assert engine.last_error is None, engine.last_error
time.sleep(round_s + 0.5)
assert engine.last_error is None, engine.last_error
print("Round replayed from card memory; parked on the looping tail.")

## 4. All one-step row/col operations


In [ ]:
GRID = 4
F_LO, F_HI = 80e6, 110e6

aod = AODSettings(
    f_min_v=F_LO,
    f_max_v=F_HI,
    f_min_h=F_LO,
    f_max_h=F_HI,
    grid_rows=GRID,
    grid_cols=GRID,
)

ROW_HZ = tuple(aod.f_min_v + i * aod.f_spacing_v for i in range(aod.grid_rows))
COL_HZ = tuple(aod.f_min_h + j * aod.f_spacing_h for j in range(aod.grid_cols))
AMP_PCT = 20.0 / GRID  # split the per-channel budget across n tones

print(
    f"{GRID}x{GRID}  Δf={aod.f_spacing_v / 1e6:.2f} MHz/site  "
    f"{AMP_PCT:.1f}%/tone\n"
    f"  rows {[f / 1e6 for f in ROW_HZ]} MHz\n"
    f"  cols {[f / 1e6 for f in COL_HZ]} MHz"
)


def grid_batch(row_from, row_to, col_from, col_to, duration_s, amp_pct=AMP_PCT):
    ramps = [
        RFRamp(
            channel=0,
            f_start=row_from[i],
            f_end=row_to[i],
            amplitude_pct=amp_pct,
            tone_index=i,
        )
        for i in range(GRID)
    ] + [
        RFRamp(
            channel=1,
            f_start=col_from[j],
            f_end=col_to[j],
            amplitude_pct=amp_pct,
            tone_index=j,
        )
        for j in range(GRID)
    ]
    return AWGBatch(ramps=ramps, travel_duration_s=duration_s)


def _one_step_dest(index, n):
    return index + 1 if index + 1 < n else index - 1


def _shifted(home, index, dest):
    out = list(home)
    out[index] = home[dest]
    return tuple(out)


ONE_STEP_OPS = [("row", i, _one_step_dest(i, GRID)) for i in range(GRID)] + [
    ("col", j, _one_step_dest(j, GRID)) for j in range(GRID)
]


def one_step_row_col_round(step_s):
    home_r, home_c = ROW_HZ, COL_HZ
    batches = [grid_batch(home_r, home_r, home_c, home_c, step_s)]  # settle
    for axis, src, dest in ONE_STEP_OPS:
        if axis == "row":
            batches.append(
                grid_batch(home_r, _shifted(home_r, src, dest), home_c, home_c, step_s)
            )
        else:
            batches.append(
                grid_batch(home_r, home_r, home_c, _shifted(home_c, src, dest), step_s)
            )
    batches.append(grid_batch(home_r, home_r, home_c, home_c, step_s))  # park
    return batches


for axis, src, dest in ONE_STEP_OPS:
    home = ROW_HZ if axis == "row" else COL_HZ
    print(f"  {axis} {src}  {home[src] / 1e6:.1f} → {home[dest] / 1e6:.1f} MHz")

In [ ]:
fs = open_engine(AWGEngineConfig(mode="stream"), F_LO, F_HI, grid=GRID)
print(f"ring = {engine.look_ahead_s * 1e3:.1f} ms of look-ahead")

batches = one_step_row_col_round(step_s=1)
n_moving = sum(1 for b in batches if any(r.f_start != r.f_end for r in b.ramps))
print(f"{len(batches)} batches ({n_moving} moving), {len(batches[0].ramps)} ramps each")

engine.load_round(batches)
print(f"loaded {engine.total_travel_duration_s:.2f} s of waveform")

print("Playing -- n tones/channel; one tone steps one site, then drops back.")
engine.play()
assert engine.last_error is None, engine.last_error

time.sleep(engine.total_travel_duration_s + 1.0)
assert engine.last_error is None, engine.last_error
print("One-step row/col sweep complete; grid parked at its start sites until close().")

## 5. Cleanup


In [ ]:
if engine is not None:
    engine.close()
    engine = None
    print("Engine closed.")